# Сколько точек нужно, чтобы предсказать следующий товар

Генеративная постановка (диффузия, flow matching) имеет смысл, если распределение следующего
товара p(i | h) **не сводится к одной точке**: точечный предсказатель регрессирует в среднее и
попадает между модами. Проверяем это напрямую, без диффузии.

**Главный тест (раздел 3).** Носитель распределения сжимаем в k центроидов, делим 10 слотов
выдачи между ними пропорционально массам и считаем HR@10. У лестницы есть оба конца:

* k = 1 — точечный предсказатель, центр масс;
* k = TOP_N в коллаборативном пространстве — выдача совпадает с top-10 самого SASRec.

Поэтому результат читается как «одна точка восстанавливает X% качества полного распределения».

**Три источника p(i | h):**

| источник | носитель | чем задана похожесть истории |
|---|---|---|
| SASRec | top-N товаров с вероятностями | — (модель) |
| группы по тексту | следующие товары 19 соседей | косинус эмбеддинга последнего товара |
| группы по состоянию | следующие товары 19 соседей | косинус скрытого состояния h SASRec |

Третий источник добавлен потому, что «похожая история» по одному последнему товару —
слабое условие: разброс следующих товаров смешивает многомодальность p(i | h) с
недообусловленностью. Состояние h — это то, что похожей историей считает сама модель.

**Два пространства.** Всё считается и в текстовом пространстве (Qwen3-Embedding-0.6B, 1024d),
и в коллаборативном (обученные эмбеддинги товаров SASRec). Кластеризация и ретривал всегда
идут в одном и том же пространстве — иначе сравнение бессмысленно.

**Контроли.** Прирост при k > 1 может объясняться не модами, а избыточностью выдачи: десять
ближайших соседей одной точки почти дубликаты друг друга. Поэтому есть MMR-бейзлайн (то же
ранжирование, но со штрафом за близость к уже выбранным) и измерение избыточности top-10.
Форма носителя сравнивается с унимодальным нулём того же разброса.

In [ ]:
import os, sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
os.chdir(ROOT)                          # пути в yaml относительны корня проекта
sys.path.insert(0, str(ROOT / 'src'))

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from scipy.cluster.hierarchy import fcluster, linkage
from scipy.spatial.distance import pdist

from builders import load_model_config
from data import build_dataloaders, load_item_embeddings

plt.rcParams['figure.dpi'] = 110
plt.rcParams['axes.spines.top']   = False
plt.rcParams['axes.spines.right'] = False
print(f'Корень проекта: {ROOT}')

In [ ]:
# ════════════════════════════════════════════════════════════
# КОНФИГ
# ════════════════════════════════════════════════════════════
DATA_DIR    = 'data/processed/toys'                            # результат scripts/preprocess.py
EMB_PATH    = f'{DATA_DIR}/qwen3-embedding-0.6b_text.pickle'   # результат scripts/embed_items.py
ITEMS_PATH  = f'{DATA_DIR}/items.parquet'
CONFIG_PATH = 'configs/toys/sasrec.yaml'
CKPT_DIR    = 'checkpoints/toys/sasrec_posloss_2'                   # базовый SASRec этого датасета

TOP_N      = 50                          # носитель модельного p(i|h)
TOP_N_GRID = (20, 50, 200)               # ось чувствительности к размеру носителя (раздел 6)
TOP_K      = 10                          # слотов выдачи (HR@K)
K_GRID     = (1, 2, 3, 5, 10, 20, 50)    # сколько точек выдаёт предсказатель
GROUP      = 50                          # группа похожих историй: якорь + GROUP-1 соседей
                                         # (50, чтобы носитель был того же размера, что top-N модели)
N_NEIGH    = 50                          # соседей на пользователя для парного анализа (раздел 4)
N_ANCHORS  = 3000                        # якорей для групп

NULL_REPEATS     = 10       # реализаций нуля на носитель (раздел 4.2)
NULL_SHRINK      = 0.2      # доля дисперсии нуля, перенесённая в изотропный шум
MMR_LAMBDA = 0.7            # MMR: вес релевантности против непохожести на уже выбранные
MMR_POOL   = 100            # кандидатов для MMR
CUT_Q      = 0.25           # порог разреза = CUT_Q-квантиль расстояний случайных пар
M_MIN      = 0.10           # значимый кластер: масса не меньше M_MIN
Q_GRID     = (0.05, 0.10, 0.25, 0.50)   # сетка чувствительности по порогу разреза
M_GRID     = (0.05, 0.10, 0.20)         # и по порогу значимости

N_TEST_USERS  = None        # ограничить число тестовых пользователей (для быстрых прогонов); None — все
CAL_USERS     = 4000        # пользователей val для калибровки температуры
CLUSTER_USERS = 3000        # пользователей для иерархической кластеризации (scipy, разделы 4, 6)
SEED          = 42

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
rng = np.random.default_rng(SEED)
print(f'Device: {device}')

## 0. Загрузка

Эмбеддинги (нормированные), названия товаров, сплиты и чекпойнт SASRec. Номер строки в
`items.parquet` — это и id товара в сплитах, и строка матрицы эмбеддингов, поэтому сверяем
их количество.

`SPACES` — два пространства, в которых дальше всё считается: текстовое и коллаборативное
(эмбеддинги товаров SASRec, нормированные, — в них же модель ранжирует, `retrieval_metric: cosine`).

In [ ]:
spec, cfg = load_model_config(CONFIG_PATH)

E = load_item_embeddings(EMB_PATH)
E /= np.linalg.norm(E, axis=1, keepdims=True)
N_ITEMS = PAD = E.shape[0]                      # id паддинга = число товаров
Et = torch.from_numpy(E).to(device)

items  = pd.read_parquet(ITEMS_PATH, columns=['parent_asin', 'text'])
TITLES = items['text'].str.split('\n').str[0].str.removeprefix('title: ').to_numpy()
assert len(items) == N_ITEMS, 'эмбеддинги и items.parquet разошлись — пересчитайте embed_items.py'

splits = {s: pd.read_pickle(Path(DATA_DIR) / f'{s}_data.df') for s in ('train', 'val', 'test')}
all_df = pd.concat([d.assign(split=s) for s, d in splits.items()], ignore_index=True)

model = spec.build(cfg).to(device).eval()
ckpt = torch.load(Path(CKPT_DIR) / 'best.pt', map_location='cpu', mmap=True)
model.load_state_dict(ckpt['model_state'])
diff = {k: (v, model.config().get(k)) for k, v in (ckpt.get('model_cfg') or {}).items()
        if model.config().get(k) != v}
W_items = F.normalize(model._item_table().detach(), dim=-1)

SPACES = {'текст': Et, 'коллаб': W_items}       # кластеризация и ретривал — всегда в одном из них

loaders = build_dataloaders(data_dir=DATA_DIR, batch_size=512, num_workers=0,
                            seq_len=cfg.data.max_seq_len, padding_idx=PAD, splits=('val', 'test'))
if N_TEST_USERS is not None:        # согласованно урезаем и таблицу, и лоадер
    splits['test'] = splits['test'].iloc[:N_TEST_USERS].reset_index(drop=True)
    loaders['test'] = torch.utils.data.DataLoader(
        torch.utils.data.Subset(loaders['test'].dataset, range(N_TEST_USERS)), batch_size=512)
    print(f'тест урезан до {N_TEST_USERS:,} пользователей (N_TEST_USERS)')

print(f'товаров {N_ITEMS:,} | текст {E.shape[1]}d, коллаб {W_items.shape[1]}d | '
      + ', '.join(f'{s} {len(d):,}' for s, d in splits.items()))
print(f'{CKPT_DIR}/best.pt | эпоха {ckpt["epoch"]} | val: '
      + ', '.join(f'{k} {v:.4f}' for k, v in ckpt['metrics'].items())
      + (f'\nОТЛИЧИЯ конфига от чекпойнта (в чекпойнте → в конфиге): {diff}' if diff else ''))

## 1. Масштаб расстояний

Пороги кластеризации выражены квантилями расстояний между случайными парами товаров, и своим
для каждого пространства: масштабы у текстового и коллаборативного пространства разные.

In [ ]:
last_all = np.array([s[-1] for s in all_df['seq']])
next_all = all_df['next'].to_numpy()
i, j = rng.integers(0, N_ITEMS, (2, 200_000)); keep = i != j

D_RAND, D_TRANS = {}, {}
for name, V in SPACES.items():
    Vn = V.cpu().numpy()
    D_RAND[name]  = 1 - (Vn[i[keep]] * Vn[j[keep]]).sum(1)
    D_TRANS[name] = 1 - (Vn[last_all] * Vn[next_all]).sum(1)


def cut_at(q: float, space: str = 'текст') -> float:
    """Порог разреза как q-квантиль расстояний случайных пар в данном пространстве."""
    return float(np.quantile(D_RAND[space], q))


D_CUT = {name: cut_at(CUT_Q, name) for name in SPACES}
qs = [0.01, 0.05, 0.25, 0.50, 0.75]
rows = {}
for name in SPACES:
    rows[f'{name}: случайные пары'] = np.quantile(D_RAND[name], qs)
    rows[f'{name}: последний → следующий'] = np.quantile(D_TRANS[name], qs)
display(pd.DataFrame(rows, index=[f'q{int(q * 100):02d}' for q in qs]).T.round(3))
for name in SPACES:
    print(f'{name}: D_CUT = {D_CUT[name]:.3f}; шаг пользователя короче D_CUT в '
          f'{np.mean(D_TRANS[name] < D_CUT[name]):.0%} переходов')

fig, axes = plt.subplots(1, 2, figsize=(11, 3.2))
for ax, name in zip(axes, SPACES):
    bins = np.linspace(0, max(D_RAND[name].max(), D_TRANS[name].max()), 80)
    ax.hist(D_RAND[name], bins, density=True, alpha=0.6, color='grey', label='случайные пары')
    ax.hist(D_TRANS[name], bins, density=True, alpha=0.6, color='steelblue', label='последний → следующий')
    for q in Q_GRID:
        ax.axvline(cut_at(q, name), color='tomato', lw=1, ls='-' if q == CUT_Q else ':')
    ax.set_title(f'{name}: масштаб расстояний', fontsize=10); ax.set_xlabel('косинусное расстояние')
axes[0].set_ylabel('плотность'); axes[0].legend(fontsize=8)
fig.tight_layout(); plt.show()

## 2. Источники p(i | h)

### 2.1 Модельный: SASRec

Температура калибруется по NLL таргета на val: обучающая τ у InfoNCE отвечает за разделение
позитива от 64 негативов, а не за остроту полного софтмакса по каталогу. Ранжирование от τ не
зависит, но от неё зависят массы, а значит и веса в кластеризации.

Полную матрицу «пользователи × товары» не храним (1.9 ГБ): наружу идут только top-N,
масса top-N, exp(H) и скрытое состояние h (оно нужно для третьего источника).

In [ ]:
@torch.no_grad()
def cosines(loader, limit=None):
    """Косинусы пользователь × товар для подвыборки сплита — только для калибровки τ."""
    cos, tgt = [], []
    for b in loader:
        h = F.normalize(model(b['history'].to(device), b['mask'].to(device)), dim=-1)
        cos.append(h @ W_items.T); tgt.append(b['target'].to(device))
        if limit is not None and sum(len(t) for t in tgt) >= limit:
            break
    return torch.cat(cos)[:limit], torch.cat(tgt)[:limit]


cos_cal, y_cal = cosines(loaders['val'], CAL_USERS)
taus = np.geomspace(0.01, 0.5, 60)
ar = torch.arange(len(y_cal), device=device)
nll_val = np.array([(torch.logsumexp(cos_cal / t, 1) - cos_cal[ar, y_cal] / t).mean().item()
                    for t in taus])
TAU_TRAIN, TAU = model.temperature, float(taus[nll_val.argmin()])
del cos_cal

fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(taus, nll_val, color='steelblue')
ax.axvline(TAU, color='seagreen', ls='--', label=f'τ* = {TAU:.3f}')
ax.axvline(TAU_TRAIN, color='grey', ls=':', label=f'τ обучения = {TAU_TRAIN}')
ax.set_xscale('log'); ax.set_xlabel('τ'); ax.set_ylabel('NLL таргета (val)'); ax.legend()
ax.set_title(f'Калибровка по {len(y_cal):,} пользователям val: NLL {nll_val.min():.3f} '
             f'против {np.log(N_ITEMS):.3f} у равномерного', fontsize=9)
fig.tight_layout(); plt.show()

In [ ]:
@torch.no_grad()
def model_support(loader, top_n: int, tau: float) -> dict:
    """top-N товаров с вероятностями, служебные числа и скрытое состояние h."""
    out = {k: [] for k in ('items', 'probs', 'mass', 'eff_n', 'target', 'hist')}
    hs = []
    for b in loader:
        hist, mask = b['history'].to(device), b['mask'].to(device)
        h = F.normalize(model(hist, mask), dim=-1)
        logp = (h @ W_items.T / tau).log_softmax(-1)
        p = logp.exp()
        pv, pi = p.topk(top_n, dim=-1)
        out['items'].append(pi.cpu().numpy()); out['probs'].append(pv.cpu().numpy())
        out['mass'].append(pv.sum(-1).cpu().numpy())
        out['eff_n'].append((-(p * logp).sum(-1)).exp().cpu().numpy())
        out['target'].append(b['target'].numpy()); out['hist'].append(b['history'].numpy())
        hs.append(h)
    res = {k: np.concatenate(v) for k, v in out.items()}
    res['h'] = torch.cat(hs)
    return res


S = model_support(loaders['test'], TOP_N, TAU)
S['hit'] = (S['items'] == S['target'][:, None]).any(1)
SASREC_HR = float((S['items'][:, :TOP_K] == S['target'][:, None]).any(1).mean())   # верхний конец лестницы
print(f'тест: {len(S["target"]):,} пользователей | таргет в top-{TOP_N}: {S["hit"].mean():.3f} | '
      f'HR@{TOP_K} самого SASRec: {SASREC_HR:.4f}')
display(pd.DataFrame({f'масса top-{TOP_N}': S['mass'], 'exp(H), товаров': S['eff_n']})
          .describe(percentiles=[0.1, 0.5, 0.9]).T[['mean', '10%', '50%', '90%']].round(3))

### 2.2 Из данных: группы похожих историй

Группа — якорь и `GROUP − 1` ближайших пользователей; носитель p(i | h) — фактические следующие
товары **соседей** с равными весами, проверяется следующий товар **якоря** (leave-one-out:
иначе центроиды строились бы по тем же товарам, попадание в которые и проверяется).

Похожесть историй задаётся двумя способами:

* **по тексту** — косинус эмбеддинга последнего товара. Последний товар, а не среднее
  нескольких: у text-only kNN среднее истории даёт HR@10 втрое хуже последнего товара;
* **по состоянию** — косинус скрытого состояния h SASRec, то есть похожесть в понимании самой
  модели. Это более сильное условие: h учитывает всю историю и порядок.

Контроль: доля пар-соседей с общим товаром в истории. Если она высока, «похожие истории» —
это в основном пересекающиеся истории, и разброс следующих товаров объясняется повторами.

In [ ]:
test_df = splits['test']
last_test = np.array([s[-1] for s in test_df['seq']])
next_test = test_df['next'].to_numpy()
hist_sets = [set(s) for s in test_df['seq']]
n_users = len(test_df)


def knn(Q: torch.Tensor, k: int, chunk: int = 2048):
    """Ближайшие соседи по косинусу среди строк Q (сам себе не сосед)."""
    sims, idxs = [], []
    for s in range(0, len(Q), chunk):
        sim = Q[s:s + chunk] @ Q.T
        rows = torch.arange(sim.shape[0], device=Q.device)
        sim[rows, torch.arange(s, s + sim.shape[0], device=Q.device)] = -2
        v, ix = sim.topk(k, dim=1)
        sims.append(v.cpu().numpy()); idxs.append(ix.cpu().numpy())
    return np.vstack(sims), np.vstack(idxs)


anchors = rng.choice(n_users, size=min(N_ANCHORS, n_users), replace=False)
NEIGH = {'текст': knn(Et[torch.from_numpy(last_test).to(device)], max(N_NEIGH, GROUP - 1)),
         'состояние': knn(F.normalize(S['h'], dim=-1), max(N_NEIGH, GROUP - 1))}

GROUPS = {}
for name, (nn_sim, nn_idx) in NEIGH.items():
    members = np.hstack([anchors[:, None], nn_idx[anchors, :GROUP - 1]])
    shared = np.mean([len(hist_sets[a] & hist_sets[b]) > 0
                      for a, row in zip(anchors, members[:, 1:]) for b in row])
    GROUPS[name] = {
        'items': next_test[members[:, 1:]],                       # носитель — соседи без якоря
        'probs': np.full((len(anchors), GROUP - 1), 1 / (GROUP - 1), np.float32),
        'target': next_test[anchors], 'members': members,
        'tightness': nn_sim[anchors, :GROUP - 1].mean(1), 'shared': shared,
        'space': 'текст' if name == 'текст' else 'коллаб',
    }
    print(f'группы по «{name}»: плотность медиана {np.median(GROUPS[name]["tightness"]):.3f} | '
          f'общий товар в истории у {shared:.1%} пар-соседей | '
          f'тот же следующий товар, что у якоря: '
          f'{np.mean(next_test[members[:, 1:]] == next_test[members[:, :1]]):.2%}')

rand_b = rng.integers(0, n_users, (len(anchors), GROUP - 1))
print(f'для сравнения, случайные пары: общий товар у '
      f'{np.mean([len(hist_sets[a] & hist_sets[b]) > 0 for a, row in zip(anchors, rand_b) for b in row]):.1%}')

## 3. Главный тест: одна точка против k точек

Процедура для каждого источника и каждого пространства:

1. взвешенный k-means на носителе, k из `K_GRID`;
2. массы кластеров = суммы весов;
3. `TOP_K` слотов делятся между центроидами пропорционально массам (наибольшие остатки);
4. для каждого центроида ближайшие товары каталога, склейка в список длины `TOP_K`;
5. HR@`TOP_K`: попал ли настоящий следующий товар (у групп — товар якоря, его в носителе нет).

**Верхний конец лестницы.** В коллаборативном пространстве при k = TOP_N каждый товар носителя
становится своим кластером, слоты достаются `TOP_K` самым вероятным, а ближайший к товару
объект — он сам. То есть выдача сходится к top-10 самого SASRec, и лестница меряет, какую долю
качества полного распределения восстанавливают k точек.

Оговорка: это точка схождения, а не потолок. Центроид усредняет носитель и может попасть в
товар, которого в top-10 модели не было, поэтому k = 1 иногда оказывается выше k = TOP_N.
Само по себе это тоже результат: сглаживание распределения работает не хуже сырого ранжирования.

In [ ]:
def kmeans_weighted(X: torch.Tensor, w: torch.Tensor, k: int, iters: int = 15, seed: int = SEED):
    """
    Взвешенный k-means на нормированных эмбеддингах, батчево по пользователям.

    Args:
        X: (U, n, d) носитель; w: (U, n) веса.
    Returns:
        centroids (U, k, d) нормированные, masses (U, k) доли массы, assign (U, n) метки.
    """
    U, n, d = X.shape
    k = min(k, n)
    g = torch.Generator(device=X.device).manual_seed(seed)
    gumbel = -torch.log(-torch.log(torch.rand(U, n, device=X.device, generator=g).clamp_min(1e-12)))
    start = (w.clamp_min(1e-12).log() + gumbel).topk(k, dim=-1).indices
    C = torch.gather(X, 1, start[..., None].expand(-1, -1, d)).clone()
    heaviest = X[torch.arange(U, device=X.device), w.argmax(-1)]
    assign = torch.zeros(U, n, dtype=torch.long, device=X.device)
    mass = w.new_zeros(U, k)
    for _ in range(iters):
        assign = torch.einsum('und,ukd->unk', X, C).argmax(-1)
        hot = F.one_hot(assign, k).to(X.dtype) * w[..., None]
        mass = hot.sum(1)
        C = torch.einsum('unk,und->ukd', hot, X) / mass.clamp_min(1e-12)[..., None]
        C = torch.where((mass <= 0)[..., None], heaviest[:, None, :].expand(-1, k, -1), C)
        C = F.normalize(C, dim=-1)
    return C, mass / mass.sum(-1, keepdim=True).clamp_min(1e-12), assign


def allocate_slots(mass: torch.Tensor, total: int) -> torch.Tensor:
    """Делит total слотов между кластерами пропорционально массам (наибольшие остатки)."""
    raw = mass * total
    base = raw.floor().long()
    rest = total - base.sum(-1)
    rank = (raw - base).argsort(-1, descending=True).argsort(-1)
    return base + (rank < rest[:, None]).long()


def merge_lists(cand: np.ndarray, slots: np.ndarray, order: np.ndarray, top_k: int) -> np.ndarray:
    """Склейка выдач центроидов: тяжёлые первыми, дубликаты пропускаются, добор до top_k."""
    out = np.empty((len(cand), top_k), dtype=np.int64)
    for b in range(len(cand)):
        picked, seen = [], set()
        for c in order[b]:
            for it in cand[b, c][:slots[b, c]]:
                if it not in seen:
                    seen.add(it); picked.append(it)
        for c in order[b]:
            for it in cand[b, c]:
                if len(picked) >= top_k:
                    break
                if it not in seen:
                    seen.add(it); picked.append(it)
        out[b] = picked[:top_k]
    return out


@torch.no_grad()
def predict_topk(src: dict, k: int, space: str, top_k: int = TOP_K, chunk: int = 512,
                 return_extras: bool = False):
    """Выдача из k центроидов носителя в заданном пространстве: (U, top_k)."""
    V = SPACES[space]
    outs, reds = [], []
    for s in range(0, len(src['items']), chunk):
        idx = torch.from_numpy(src['items'][s:s + chunk]).to(device)
        w = torch.from_numpy(src['probs'][s:s + chunk]).to(device).float()
        C, mass, _ = kmeans_weighted(V[idx], w, k)
        slots = allocate_slots(mass, top_k)
        cand = torch.einsum('bkd,nd->bkn', C, V).topk(top_k, dim=-1).indices.cpu().numpy()
        order = torch.argsort(mass, dim=-1, descending=True).cpu().numpy()
        picked = merge_lists(cand, slots.cpu().numpy(), order, top_k)
        outs.append(picked)
        if return_extras:                                   # избыточность выдачи
            P = V[torch.from_numpy(picked).to(device)]
            sim = torch.einsum('bkd,bmd->bkm', P, P)
            off = ~torch.eye(top_k, dtype=torch.bool, device=device)
            reds.append(sim[:, off].mean(-1).cpu().numpy())
    pred = np.vstack(outs)
    return (pred, np.concatenate(reds)) if return_extras else pred


def hr(pred: np.ndarray, target: np.ndarray) -> float:
    return float((pred == target[:, None]).any(1).mean())

In [ ]:
SOURCES = {f'SASRec top-{TOP_N}': S,
           'группы по тексту': GROUPS['текст'],
           'группы по состоянию': GROUPS['состояние']}

rows, preds = {}, {}
for src_name, src in SOURCES.items():
    for space in SPACES:
        key = f'{src_name} / {space}'
        preds[key] = {k: predict_topk(src, k, space) for k in K_GRID}
        rows[key] = {f'k={k}': hr(preds[key][k], src['target']) for k in K_GRID}
ladder = pd.DataFrame(rows).T
ladder['k=3 / k=1'] = ladder['k=3'] / ladder['k=1'].replace(0, np.nan)
ladder[f'% от SASRec HR@{TOP_K}'] = 100 * ladder['k=1'] / SASREC_HR
print(f'HR@{TOP_K} от числа точек k. Верхний конец: HR@{TOP_K} самого SASRec = {SASREC_HR:.4f}')
display(ladder.round(4))

fig, ax = plt.subplots(figsize=(7, 3.6))
for key, style in zip(rows, ('o-', 'o--', 's-', 's--', '^-', '^--')):
    ax.plot(K_GRID, [rows[key][f'k={k}'] for k in K_GRID], style, ms=4, label=key)
ax.axhline(SASREC_HR, color='tomato', ls=':', lw=1.5, label=f'SASRec top-{TOP_K}')
ax.set_xscale('log'); ax.set_xticks(K_GRID); ax.set_xticklabels(K_GRID)
ax.set_xlabel('число точек k'); ax.set_ylabel(f'HR@{TOP_K}'); ax.legend(fontsize=7)
ax.set_title('Лестница: от одной точки до полного распределения'); fig.tight_layout(); plt.show()

### Где возникает выигрыш

Отношение `k=3 / k=1` и абсолютный прирост ведут себя по-разному: относительный растёт с
неопределённостью, абсолютный падает. Для практической пользы важен абсолютный, поэтому
приводим обе колонки.

In [ ]:
MAIN_SPACE = 'коллаб'                       # пространство, в котором модель ранжирует
p_main = preds[f'SASRec top-{TOP_N} / {MAIN_SPACE}']
quart = pd.qcut(S['eff_n'], 4, labels=[f'Q{i}' for i in range(1, 5)])
rows = {}
for q in quart.categories:
    m = np.asarray(quart == q)
    r = {f'k={k}': hr(p_main[k][m], S['target'][m]) for k in (1, 2, 3, 5)}
    r['абс. прирост k=3 − k=1'] = r['k=3'] - r['k=1']
    r['отн. k=3 / k=1'] = r['k=3'] / r['k=1'] if r['k=1'] else np.nan
    r['пользователей'] = int(m.sum())
    rows[f'{q}: exp(H) {S["eff_n"][m].min():.0f}–{S["eff_n"][m].max():.0f}'] = r
display(pd.DataFrame(rows).T.round(4))

### Контроль 1: не сводится ли выигрыш к диверсификации

Десять ближайших соседей одной точки почти дубликаты друг друга, и любое разнообразие
повышает шанс попадания. MMR берёт ранжирование **одной** точки (k = 1) и набирает 10 слотов
со штрафом за близость к уже выбранным. Если MMR забирает большую часть прироста, дело в
избыточности выдачи, а не в модах.

In [ ]:
@torch.no_grad()
def predict_mmr(src: dict, space: str, lam: float = MMR_LAMBDA, pool: int = MMR_POOL,
                top_k: int = TOP_K, chunk: int = 512) -> np.ndarray:
    """MMR поверх ранжирования одной точки: λ·релевантность − (1−λ)·максимум сходства с выбранными."""
    V = SPACES[space]
    out = []
    for s in range(0, len(src['items']), chunk):
        idx = torch.from_numpy(src['items'][s:s + chunk]).to(device)
        w = torch.from_numpy(src['probs'][s:s + chunk]).to(device).float()
        C, _, _ = kmeans_weighted(V[idx], w, 1)
        scores = torch.einsum('bkd,nd->bn', C, V)
        rel, cand = scores.topk(pool, dim=-1)                      # (B, pool)
        Vc = V[cand]                                               # (B, pool, d)
        sim = torch.einsum('bpd,bqd->bpq', Vc, Vc)
        B = len(rel)
        chosen = torch.zeros(B, top_k, dtype=torch.long, device=device)
        best_sim = torch.full((B, pool), -1.0, device=device)
        taken = torch.zeros(B, pool, dtype=torch.bool, device=device)
        for t in range(top_k):
            score = lam * rel - (1 - lam) * best_sim
            score[taken] = -1e9
            pick = score.argmax(-1)
            chosen[:, t] = cand[torch.arange(B, device=device), pick]
            taken[torch.arange(B, device=device), pick] = True
            best_sim = torch.maximum(best_sim, sim[torch.arange(B, device=device), pick])
        out.append(chosen.cpu().numpy())
    return np.vstack(out)


rows = {}
for src_name, src in SOURCES.items():
    space = 'коллаб' if 'SASRec' in src_name or src['space'] == 'коллаб' else 'текст'
    p1, red1 = predict_topk(src, 1, space, return_extras=True)
    p3, red3 = predict_topk(src, 3, space, return_extras=True)
    mmr = predict_mmr(src, space)
    base, full = hr(p1, src['target']), hr(p3, src['target'])
    rows[f'{src_name} / {space}'] = {
        'k=1': base, 'MMR поверх k=1': hr(mmr, src['target']), 'k=3': full,
        'доля прироста у MMR': (hr(mmr, src['target']) - base) / (full - base) if full > base else np.nan,
        'избыточность top-10 при k=1': red1.mean(), 'при k=3': red3.mean()}
print('MMR-контроль: сколько прироста объясняется разнообразием выдачи, а не модами')
display(pd.DataFrame(rows).T.round(4))

### Контроль 2: оракул выбора моды

Если бы предсказатель всегда угадывал, **в какой** моде лежит ответ, он мог бы отдать все 10
слотов ей одной. Считаем это для пользователей, у которых таргет есть в носителе: находим
кластер с таргетом и выдаём 10 ближайших к его центроиду.

Разрыв между k = 1 и оракулом — это и есть величина, которую может отыграть генеративная
модель, умеющая выбирать моду. Оракул считается и на всех пользователях (у кого таргета в
носителе нет, попадание невозможно), и отдельно на тех, у кого он есть.

In [ ]:
@torch.no_grad()
def oracle_hr(src: dict, k: int, space: str, top_k: int = TOP_K, chunk: int = 512) -> dict:
    """Верхний предел: слоты отданы кластеру, в котором лежит таргет."""
    V = SPACES[space]
    hit, in_support = [], []
    for s in range(0, len(src['items']), chunk):
        idx_np = src['items'][s:s + chunk]
        tgt = torch.from_numpy(src['target'][s:s + chunk]).to(device)
        idx = torch.from_numpy(idx_np).to(device)
        w = torch.from_numpy(src['probs'][s:s + chunk]).to(device).float()
        C, _, assign = kmeans_weighted(V[idx], w, k)
        pos = (idx == tgt[:, None])                               # где таргет в носителе
        has = pos.any(-1)
        # Первое вхождение: в носителе групп товар может повторяться, и суммой меток пользоваться нельзя
        cl = assign.gather(1, pos.float().argmax(-1)[:, None]).squeeze(1)
        Csel = C[torch.arange(len(C), device=device), cl]         # (B, d)
        top = (Csel @ V.T).topk(top_k, dim=-1).indices
        hit.append(((top == tgt[:, None]).any(-1) & has).cpu().numpy())
        in_support.append(has.cpu().numpy())
    hit, in_support = np.concatenate(hit), np.concatenate(in_support)
    return {'оракул, все пользователи': hit.mean(),
            'оракул, таргет в носителе': hit[in_support].mean() if in_support.any() else np.nan,
            'доля с таргетом в носителе': in_support.mean()}


rows = {}
for src_name, src in SOURCES.items():
    space = 'коллаб' if 'SASRec' in src_name or src['space'] == 'коллаб' else 'текст'
    base = hr(predict_topk(src, 1, space), src['target'])
    for k in (2, 3, 5):
        rows[f'{src_name} / {space}, k={k}'] = {'k=1 (точка)': base, **oracle_hr(src, k, space)}
oracle = pd.DataFrame(rows).T
oracle['запас: оракул − k=1'] = oracle['оракул, все пользователи'] - oracle['k=1 (точка)']
display(oracle.round(4))

## 4. Форма носителя

### 4.1 Куда идут пользователи с похожими историями

Для пар-соседей с высокой близостью историй смотрим косинус между их следующими товарами.
Унимодальное узкое p(i | h) прижало бы распределение к 1; смесь разнесённых мод даёт часть
пар на уровне фона случайных пар. Сравниваем оба определения похожести: по тексту (последний
товар) и по состоянию h.

In [ ]:
a_r, b_r = rng.integers(0, n_users, (2, 200_000)); keep_r = a_r != b_r


def next_cos(a, b, space='текст', chunk=200_000):
    V = SPACES[space].cpu().numpy()
    return np.concatenate([(V[next_test[a[s:s + chunk]]] * V[next_test[b[s:s + chunk]]]).sum(1)
                           for s in range(0, len(a), chunk)])


fig, axes = plt.subplots(1, 2, figsize=(12, 3.4))
for ax, (name, (nn_sim, nn_idx)) in zip(axes, NEIGH.items()):
    space = 'текст' if name == 'текст' else 'коллаб'
    a_nn = np.repeat(np.arange(n_users), N_NEIGH); b_nn = nn_idx[:, :N_NEIGH].ravel()
    sim_nn = nn_sim[:, :N_NEIGH].ravel()
    bg = next_cos(a_r[keep_r], b_r[keep_r], space)
    bins = np.linspace(-0.2, 1.0, 70)
    ax.hist(bg, bins, density=True, color='grey', alpha=0.55, label='случайные пары')
    for thr, color in ((0.95, 'steelblue'), (0.99, 'darkorange')):
        m = sim_nn > thr
        if m.sum() < 50:
            continue
        ax.hist(next_cos(a_nn[m], b_nn[m], space), bins, density=True, histtype='step', lw=1.8,
                color=color, label=f'близость историй > {thr} ({m.sum():,} пар)')
    ax.set_title(f'похожесть по «{name}», cos в пространстве «{space}»', fontsize=10)
    ax.set_xlabel('cos между следующими товарами'); ax.legend(fontsize=7)
    if name == 'текст':
        TEXT_PAIRS = (a_nn, b_nn, sim_nn)
axes[0].set_ylabel('плотность'); fig.tight_layout(); plt.show()

### 4.2 Есть ли вообще моды

Четыре проверки, каждая отвечает на свой вопрос.

**Нуль.** Носитель сравнивается с унимодальной гауссианой, у которой та же взвешенная
ковариация: совпадают разброс, вытянутость и главные компоненты, поэтому форма облака перестаёт
быть отличием и остаётся только многомодальность. Изотропный шар для этой роли не годится: у
вытянутого облака k-means режет его по длинной оси и даёт разнесённые центроиды без всяких мод.

Ковариация оценивается по n точкам в 128 или 1024 измерениях, то есть вырождена (ранг ≤ n − 1).
Поэтому у нуля есть две поправки:

* `NULL_SHRINK` — сжатие к изотропному (Ledoit-Wolf в простейшем виде): часть дисперсии
  переносится в изотропный шум, иначе вся масса нуля уходит в несколько направлений и он
  выглядит кластеризованнее данных;
* нуль строится **на сфере**, в касательном пространстве центра масс: углы до центра берутся
  из носителя (те же значения в случайном порядке), направления — из гауссианы с ковариацией
  касательных направлений носителя. Гауссиана в объемлющем пространстве с последующей нормировкой
  не годится: нормировка искажает углы, и нуль выходит заметно компактнее данных.

Диагностика ниже показывает, что нуль действительно воспроизводит носитель: средний попарный
косинус и первые собственные значения ковариации должны совпадать.

**Усреднение и попарность.** Одна реализация нуля шумит, поэтому берём `NULL_REPEATS` реализаций
и усредняем. Сравнение попарное, по каждому пользователю отдельно: в таблице доля пользователей,
у которых носитель разделён сильнее своего нуля, с доверительным интервалом — среднее по тысячам
пользователей скрывает, идёт ли эффект от всех или от небольшой части.

**Провал плотности.** Разнесённые центроиды — косвенный признак. Прямой признак смеси —
разрежение между пиками: проецируем носитель на прямую между двумя центроидами и считаем
отношение массы в средней полосе к средней массе в полосах у центроидов. У смеси отношение
меньше 1 (провал), у одного облака любой формы провала нет.

**Популярностный приор.** Двумодальность может быть не про второй интерес, а про то, что модель
подмешивает в top-N глобально популярные товары. Для каждого кластера считаем среднюю
популярность товаров по train и среднее расстояние до последнего товара истории: если один
кластер близок к истории и непопулярен, а второй далёк и популярен — механизм именно такой.

In [ ]:
def mean_pair_cos(X: torch.Tensor) -> torch.Tensor:
    G = torch.einsum('und,umd->unm', X, X)
    off = ~torch.eye(X.shape[1], dtype=torch.bool, device=X.device)
    return G[:, off].mean(-1)


def matched_null(X: torch.Tensor, w: torch.Tensor, seed: int = SEED,
                 shrink: float = NULL_SHRINK) -> torch.Tensor:
    """
    Унимодальный нуль на сфере: те же углы до центра носителя и та же анизотропия.

    Строится в касательном пространстве в центре масс c:
      * угол до центра θᵢ = arccos(c·xᵢ) берётся из носителя (те же значения, случайно
        переставленные) — товары каталога лежат на сфере, и радиальное распределение должно
        совпадать, иначе «кучка у центра» читается как кластер;
      * направление — гауссиана с ковариацией касательных направлений носителя
        (A = √w ⊙ uᵢ, точка = Aᵀa при a ~ N(0, I)), поэтому вытянутость и главные компоненты
        воспроизводятся;
      * shrink — часть дисперсии переносится в изотропный шум: ковариация оценивается по n
        точкам в d ≫ n измерениях и вырождена (ранг ≤ n − 1), без сжатия нуль оказывается
        структурнее данных.

    Гауссиану в объемлющем пространстве с последующей нормировкой использовать нельзя:
    нормировка искажает углы, и нуль получается заметно компактнее носителя.
    """
    U, n, d = X.shape
    g = torch.Generator(device=X.device).manual_seed(seed)
    c = F.normalize((X * w[..., None]).sum(1), dim=-1)
    ct = torch.einsum('und,ud->un', X, c).clamp(-1, 1)
    theta = ct.arccos()                                                  # углы до центра
    tang = F.normalize(X - ct[..., None] * c[:, None, :], dim=-1)        # касательные направления
    A = tang * w.clamp_min(0).sqrt()[..., None]

    Y = torch.einsum('usn,und->usd', torch.randn(U, n, n, device=X.device, generator=g), A)
    if shrink > 0:
        iso = ((A * A).sum((1, 2)) / d)[:, None, None].sqrt()
        Y = (1 - shrink) ** 0.5 * Y + (shrink ** 0.5) * iso *             torch.randn(U, n, d, device=X.device, generator=g)
    Y = Y - torch.einsum('usd,ud->us', Y, c)[..., None] * c[:, None, :]  # строго в касательное
    Y = F.normalize(Y, dim=-1)

    th = theta[:, torch.randperm(n, device=X.device, generator=g)]       # те же углы носителя
    return th.cos()[..., None] * c[:, None, :] + th.sin()[..., None] * Y


def cov_spectrum(X: torch.Tensor, w: torch.Tensor, top: int = 5) -> torch.Tensor:
    """Первые собственные значения взвешенной ковариации носителя как доли следа."""
    c = F.normalize((X * w[..., None]).sum(1), dim=-1)
    A = (X - c[:, None, :]) * w.clamp_min(0).sqrt()[..., None]
    G = torch.einsum('und,umd->unm', A, A)                        # спектр AAᵀ = спектр AᵀA
    ev = torch.linalg.eigvalsh(G).flip(-1)
    return ev[:, :top] / G.diagonal(dim1=1, dim2=2).sum(-1, keepdim=True).clamp_min(1e-12)


def separation(X: torch.Tensor, w: torch.Tensor, k: int = 2) -> dict:
    """Разделённость кластеров и провал плотности между ними."""
    C, mass, assign = kmeans_weighted(X, w, k)
    off = ~torch.eye(k, dtype=torch.bool, device=X.device)
    cent_cos = torch.einsum('ukd,uld->ukl', C, C)[:, off].mean(-1)
    same = (assign[:, :, None] == assign[:, None, :])
    pair = torch.einsum('und,umd->unm', X, X)
    eye = torch.eye(X.shape[1], dtype=torch.bool, device=X.device)
    within = (pair * (same & ~eye)).sum((1, 2)) / (same & ~eye).sum((1, 2)).clamp_min(1)
    between = (pair * ~same).sum((1, 2)) / (~same).sum((1, 2)).clamp_min(1)

    # Провал плотности: проекция на прямую между центроидами, три полосы
    axis = F.normalize(C[:, 1] - C[:, 0], dim=-1)
    t = torch.einsum('und,ud->un', X, axis)
    t0 = torch.einsum('ud,ud->u', C[:, 0], axis)
    t1 = torch.einsum('ud,ud->u', C[:, 1], axis)
    lo, hi = torch.minimum(t0, t1), torch.maximum(t0, t1)
    u = (t - lo[:, None]) / (hi - lo).clamp_min(1e-6)[:, None]
    middle = ((u >= 0.35) & (u <= 0.65))
    peaks = (u < 0.35) | (u > 0.65)
    m_mid = (w * middle).sum(-1) / 0.30                           # плотность = масса / ширина полосы
    m_peak = (w * peaks).sum(-1) / 0.70
    return {'cos центроидов': cent_cos, 'внутри − между': within - between,
            'масса 2-го кластера': mass.min(-1).values,
            'провал плотности': m_mid / m_peak.clamp_min(1e-9)}


STATS = ('cos центроидов', 'внутри − между', 'масса 2-го кластера', 'провал плотности')
# Признак многомодальности: у cos центроидов и провала — значение НИЖЕ нуля, у остальных выше
LOWER_IS_MULTIMODAL = {'cos центроидов': True, 'внутри − между': False,
                       'масса 2-го кластера': False, 'провал плотности': True}


def shape_stats(src: dict, space: str, chunk: int = 256) -> dict:
    """Статистики носителя и усреднённого по NULL_REPEATS нуля, по пользователю."""
    V = SPACES[space]
    real = {k: [] for k in STATS}
    null = {k: [] for k in STATS}
    extra = {'ср. попарный cos': [], 'ср. попарный cos (нуль)': [],
             'спектр': [], 'спектр (нуль)': []}
    for s in range(0, len(src['items']), chunk):
        X = V[torch.from_numpy(src['items'][s:s + chunk]).to(device)]
        w = torch.from_numpy(src['probs'][s:s + chunk]).to(device).float()
        for key, val in separation(X, w).items():
            real[key].append(val.cpu().numpy())
        extra['ср. попарный cos'].append(mean_pair_cos(X).cpu().numpy())
        extra['спектр'].append(cov_spectrum(X, w).cpu().numpy())
        acc = {k: [] for k in STATS}
        pair_cos_n, spec_n = [], []
        for r in range(NULL_REPEATS):
            Xn = matched_null(X, w, seed=SEED + r)
            for key, val in separation(Xn, w).items():
                acc[key].append(val)
            if r == 0:
                pair_cos_n.append(mean_pair_cos(Xn)); spec_n.append(cov_spectrum(Xn, w))
        for key in STATS:
            null[key].append(torch.stack(acc[key]).mean(0).cpu().numpy())
        extra['ср. попарный cos (нуль)'].append(pair_cos_n[0].cpu().numpy())
        extra['спектр (нуль)'].append(spec_n[0].cpu().numpy())
    out = {k: np.concatenate(v) for k, v in real.items()}
    out |= {f'{k} (нуль)': np.concatenate(v) for k, v in null.items()}
    out['ср. попарный cos'] = np.concatenate(extra['ср. попарный cos'])
    out['ср. попарный cos (нуль)'] = np.concatenate(extra['ср. попарный cos (нуль)'])
    out['спектр'] = np.vstack(extra['спектр']).mean(0)
    out['спектр (нуль)'] = np.vstack(extra['спектр (нуль)']).mean(0)
    return out


# Модельный носитель считаем в обоих пространствах: список легко расширить любой парой
SHAPE_COMBOS = [(f'SASRec top-{TOP_N}', 'коллаб'), (f'SASRec top-{TOP_N}', 'текст'),
                ('группы по тексту', 'текст'), ('группы по состоянию', 'коллаб')]
SHAPE = {f'{name} / {space}': shape_stats(SOURCES[name], space) for name, space in SHAPE_COMBOS}

In [ ]:
# Диагностика нуля: воспроизводит ли он разброс и форму носителя
rows = {}
for key, st in SHAPE.items():
    row = {'ср. попарный cos': st['ср. попарный cos'].mean(),
           'то же у нуля': st['ср. попарный cos (нуль)'].mean()}
    row.update({f'λ{i + 1}': v for i, v in enumerate(st['спектр'])})
    row.update({f'λ{i + 1} (нуль)': v for i, v in enumerate(st['спектр (нуль)'])})
    rows[key] = row
print(f'Диагностика нуля (NULL_SHRINK={NULL_SHRINK}): '
      f'λ — первые собственные значения ковариации как доли следа')
display(pd.DataFrame(rows).T.round(3))

In [ ]:
# Попарное сравнение носителя с его нулём: доля пользователей, у кого носитель «многомодальнее»
rows = {}
for key, st in SHAPE.items():
    row = {}
    for stat in STATS:
        real, null = st[stat], st[f'{stat} (нуль)']
        better = (real < null) if LOWER_IS_MULTIMODAL[stat] else (real > null)
        p = better.mean()
        half = 1.96 * np.sqrt(p * (1 - p) / len(real))
        row[f'{stat}: носитель'] = real.mean()
        row[f'{stat}: нуль'] = null.mean()
        row[f'{stat}: доля польз.'] = p
        row[f'{stat}: ±'] = half
    rows[key] = row
paired = pd.DataFrame(rows).T
print(f'Носитель против нуля ({NULL_REPEATS} реализаций, попарно по пользователям). '
      f'«Доля польз.» — у кого носитель многомодальнее своего нуля (0.5 = нет эффекта)')
for stat in STATS:
    cols = [c for c in paired.columns if c.startswith(stat)]
    print(f'\n{stat} ({"ниже" if LOWER_IS_MULTIMODAL[stat] else "выше"} у смеси):')
    display(paired[cols].round(3))

In [ ]:
# Не является ли второй кластер популярностным приором
train_items = np.concatenate([np.concatenate([np.asarray(s) for s in splits['train']['seq']]),
                              splits['train']['next'].to_numpy()])
pop = np.bincount(train_items, minlength=N_ITEMS).astype(np.float32)
pop_t = torch.from_numpy(np.log1p(pop)).to(device)
last_hist = torch.from_numpy(S['hist'][:, -1].astype(np.int64)).to(device)


@torch.no_grad()
def cluster_profile(src: dict, space: str, last: torch.Tensor, k: int = 2, chunk: int = 256) -> pd.DataFrame:
    """Для тяжёлого и второго кластера: популярность товаров и близость к последнему товару истории."""
    V = SPACES[space]
    acc = {c: {'log популярность': [], 'cos с последним товаром': [], 'масса': []} for c in (0, 1)}
    for s in range(0, len(src['items']), chunk):
        idx = torch.from_numpy(src['items'][s:s + chunk]).to(device)
        w = torch.from_numpy(src['probs'][s:s + chunk]).to(device).float()
        _, mass, assign = kmeans_weighted(V[idx], w, k)
        rank = mass.argsort(-1, descending=True).argsort(-1)       # 0 — тяжёлый кластер
        rank_of_item = rank.gather(1, assign)
        cos_last = torch.einsum('und,ud->un', V[idx], V[last[s:s + chunk]])
        for c in (0, 1):
            m = (rank_of_item == c)
            ww = w * m
            denom = ww.sum(-1).clamp_min(1e-9)
            acc[c]['log популярность'].append((ww * pop_t[idx]).sum(-1).div(denom).cpu().numpy())
            acc[c]['cos с последним товаром'].append((ww * cos_last).sum(-1).div(denom).cpu().numpy())
            acc[c]['масса'].append(mass.gather(1, rank.argsort(-1)[:, c:c + 1]).squeeze(1).cpu().numpy())
    return pd.DataFrame({('тяжёлый кластер' if c == 0 else 'второй кластер'):
                         {k: float(np.nanmean(np.concatenate(v))) for k, v in acc[c].items()}
                         for c in (0, 1)}).T


print('Профиль кластеров модельного носителя (k=2): популярность и близость к истории')
display(cluster_profile(S, MAIN_SPACE, last_hist).round(3))
print(f'для сравнения: log популярность по каталогу {np.log1p(pop).mean():.3f}, '
      f'у товаров из train-таргетов {np.log1p(pop[splits["train"]["next"]]).mean():.3f}')

### 4.3 Устойчивость к порогу разреза

Иерархическая кластеризация с разрезом на разных порогах, для носителя и для нуля того же
разброса. Смотрим и массу второго кластера, и разделённость: первая почти не отличает носитель
от нуля, вторая должна отличать.

In [ ]:
def cos_linkage(X: np.ndarray) -> np.ndarray:
    return linkage(np.maximum(pdist(X, 'cosine'), 0), method='average')


def cluster_masses(Z: np.ndarray, w: np.ndarray, d_cut: float):
    lab = fcluster(Z, t=d_cut, criterion='distance')
    mass = np.bincount(lab, weights=w)[1:] / w.sum()
    order = np.argsort(-mass)
    rank = np.empty_like(order); rank[order] = np.arange(len(order))
    return rank[lab - 1], mass[order]


def cut_stats(X: np.ndarray, w: np.ndarray, d_cut: float) -> tuple[float, float]:
    """Масса второго кластера и косинус между центрами двух тяжелейших кластеров."""
    rank, mass = cluster_masses(cos_linkage(X), w, d_cut)
    if len(mass) < 2:
        return 0.0, 1.0
    cs = []
    for r in (0, 1):
        v = (X[rank == r] * w[rank == r, None]).sum(0)
        cs.append(v / max(np.linalg.norm(v), 1e-9))
    return float(mass[1]), float(cs[0] @ cs[1])


sub = rng.choice(len(S['target']), min(CLUSTER_USERS, len(S['target'])), replace=False)
Vc = SPACES[MAIN_SPACE]
X_sub = Vc[torch.from_numpy(S['items'][sub]).to(device)]
W_sub = torch.from_numpy(S['probs'][sub]).to(device).float()
X_null = matched_null(X_sub, W_sub)
Xs, Xn = X_sub.cpu().numpy(), X_null.cpu().numpy()
Ws = (S['probs'][sub] / S['probs'][sub].sum(1, keepdims=True))

rows = {}
for q in Q_GRID:
    d = cut_at(q, MAIN_SPACE)
    real = np.array([cut_stats(Xs[u], Ws[u], d) for u in range(len(sub))])
    null = np.array([cut_stats(Xn[u], Ws[u], d) for u in range(len(sub))])
    rows[f'q{int(q * 100):02d} (d={d:.3f})'] = {
        'масса 2-го': real[:, 0].mean(), 'масса 2-го (нуль)': null[:, 0].mean(),
        'cos центроидов': real[:, 1].mean(), 'cos центроидов (нуль)': null[:, 1].mean(),
        'доля с массой ≥ M_MIN': (real[:, 0] >= M_MIN).mean()}
print(f'Иерархическая кластеризация, {len(sub):,} пользователей, пространство «{MAIN_SPACE}»')
display(pd.DataFrame(rows).T.round(3))

### 4.4 Из чего состоят совпадения у почти одинаковых историй

In [ ]:
a_nn, b_nn, sim_nn = TEXT_PAIRS
m = sim_nn > 0.99
if m.sum() >= 50:
    same_item = next_test[a_nn[m]] == next_test[b_nn[m]]
    cos_next = next_cos(a_nn[m], b_nn[m])
    near_dup = (~same_item) & (cos_next > 0.98)
    bg = next_cos(a_r[keep_r], b_r[keep_r])
    print(f'пар с близостью историй > 0.99 (по тексту): {m.sum():,}')
    display(pd.Series({'тот же следующий товар': same_item.mean(),
                       'почти-дубликат (cos > 0.98)': near_dup.mean(),
                       'разные товары': 1 - same_item.mean() - near_dup.mean()}).round(4))
    print(f'среди «разных товаров» медиана cos {np.median(cos_next[~(same_item | near_dup)]):.3f} '
          f'против {np.median(bg):.3f} у случайных пар')

## 5. Реальны ли второстепенные моды

Если p(i | h) откалибровано и второстепенные кластеры не артефакт, фактические таргеты должны
попадать в них с частотой, которую предсказывает масса. Считаем на **всех** тестовых
пользователях (k-means, батчево), а не на подвыборке: попаданий в носитель около 2.5 тысяч,
и доверительные интервалы получаются вдвое уже.

In [ ]:
@torch.no_grad()
def mode_calibration(src: dict, k: int, space: str, chunk: int = 512) -> pd.DataFrame:
    V = SPACES[space]
    cats = ['главный кластер', 'другой значимый', 'незначимые']
    obs, exp_ = np.zeros(3), np.zeros(3)
    for s in range(0, len(src['items']), chunk):
        idx = torch.from_numpy(src['items'][s:s + chunk]).to(device)
        w = torch.from_numpy(src['probs'][s:s + chunk]).to(device).float()
        tgt = torch.from_numpy(src['target'][s:s + chunk]).to(device)
        _, mass, assign = kmeans_weighted(V[idx], w, k)
        order = mass.argsort(-1, descending=True)
        rank = order.argsort(-1)                                   # 0 — самый тяжёлый
        cat_of_cluster = torch.where(rank == 0, 0, torch.where(mass >= M_MIN, 1, 2))
        support_mass = torch.from_numpy(src['probs'][s:s + chunk].sum(1)).to(device)
        for c in range(3):
            exp_[c] += ((cat_of_cluster == c).float() * mass).sum(-1).mul(support_mass).sum().item()
        pos = (idx == tgt[:, None])
        has = pos.any(-1)
        cl = assign.gather(1, pos.float().argmax(-1)[:, None]).squeeze(1)   # первое вхождение таргета
        cat = cat_of_cluster[torch.arange(len(cl), device=device), cl][has]
        obs += np.bincount(cat.cpu().numpy(), minlength=3)
    n_hit = int(obs.sum())
    p = obs / n_hit
    half = 1.96 * np.sqrt(p * (1 - p) / n_hit)
    return pd.DataFrame({'ожидаемая доля (масса)': exp_ / exp_.sum(), 'фактическая доля': p,
                         '95% ДИ снизу': p - half, '95% ДИ сверху': p + half},
                        index=cats), n_hit


calib, n_hit = mode_calibration(S, 3, MAIN_SPACE)
print(f'k = 3 кластера, пространство «{MAIN_SPACE}», попаданий в носитель: {n_hit:,} '
      f'из {len(S["target"]):,} пользователей')
display(calib.round(3))

## 6. Чувствительность

Массовые метрики (масса второго кластера, разделённость) на сетке «порог разреза × размер
носителя», рядом с унимодальным нулём того же разброса. Число кластеров как метрику не берём:
оно скачет от порога, а масса и разделённость меняются плавно.

In [ ]:
rows = []
for top_n in TOP_N_GRID:
    Sn = S if top_n == TOP_N else model_support(loaders['test'], top_n, TAU)
    pick = rng.choice(len(Sn['target']), min(CLUSTER_USERS, len(Sn['target'])), replace=False)
    Xg = Vc[torch.from_numpy(Sn['items'][pick]).to(device)]
    Wg = torch.from_numpy(Sn['probs'][pick]).to(device).float()
    Xr, Xz = Xg.cpu().numpy(), matched_null(Xg, Wg).cpu().numpy()
    Wn = Sn['probs'][pick] / Sn['probs'][pick].sum(1, keepdims=True)
    for q in Q_GRID:
        d = cut_at(q, MAIN_SPACE)
        real = np.array([cut_stats(Xr[u], Wn[u], d) for u in range(len(pick))])
        null = np.array([cut_stats(Xz[u], Wn[u], d) for u in range(len(pick))])
        row = {'TOP_N': top_n, 'порог': f'q{int(q * 100):02d}',
               'масса 2-го': real[:, 0].mean(), 'масса 2-го (нуль)': null[:, 0].mean(),
               'cos центроидов': real[:, 1].mean(), 'cos центроидов (нуль)': null[:, 1].mean()}
        row.update({f'доля массы ≥ {mm}': (real[:, 0] >= mm).mean() for mm in M_GRID})
        rows.append(row)
sens = pd.DataFrame(rows).set_index(['TOP_N', 'порог'])
display(sens.round(3))

fig, axes = plt.subplots(1, 2, figsize=(11, 3.2))
for top_n in TOP_N_GRID:
    part = sens.loc[top_n]
    axes[0].plot(range(len(part)), part['масса 2-го'], 'o-', label=f'TOP_N = {top_n}')
    axes[1].plot(range(len(part)), part['cos центроидов'], 'o-', label=f'TOP_N = {top_n}')
    axes[1].plot(range(len(part)), part['cos центроидов (нуль)'], 'o:', color='grey', lw=1)
for ax, title in zip(axes, ('масса второго кластера', 'cos между центроидами (пунктир — нуль)')):
    ax.set_xticks(range(len(Q_GRID)), [f'q{int(q * 100):02d}' for q in Q_GRID])
    ax.set_xlabel('порог разреза'); ax.set_title(title, fontsize=10); ax.legend(fontsize=7)
fig.tight_layout(); plt.show()

## 7. Примеры с названиями

Три случая: одна мода, две сопоставимые моды и — главное — случай, где **таргет лежит во втором
кластере**. Последний наглядно показывает, что теряет одна точка: центр масс уходит к тяжёлой
моде, а пользователь пошёл в лёгкую.

In [ ]:
@torch.no_grad()
def user_clusters(u: int, k: int = 3, space: str = MAIN_SPACE):
    V = SPACES[space]
    idx = torch.from_numpy(S['items'][u:u + 1]).to(device)
    w = torch.from_numpy(S['probs'][u:u + 1]).to(device).float()
    C, mass, assign = kmeans_weighted(V[idx], w, k)
    order = mass[0].argsort(descending=True).cpu().numpy()
    rank = {c: r for r, c in enumerate(order)}
    return (np.array([rank[int(c)] for c in assign[0].cpu().numpy()]),
            mass[0].cpu().numpy()[order])


def show_user(u: int, header: str, k: int = 3, max_items: int = 6) -> None:
    rank, mass = user_clusters(u, k)
    hist = S['hist'][u][S['hist'][u] != PAD]
    print(f'\n=== {header} (пользователь {u}) ===')
    print('история: ' + ' | '.join(TITLES[i][:55] for i in hist[-4:]))
    for r in range(k):
        members = np.where(rank == r)[0]
        members = members[np.argsort(-S['probs'][u][members])]
        if not len(members):
            continue
        print(f'  кластер {r} (масса {mass[r]:.2f}, товаров {len(members)}):')
        for pos in members[:max_items]:
            it = S['items'][u][pos]
            mark = '  ← таргет' if it == S['target'][u] else ''
            print(f'      p={S["probs"][u][pos]:.4f}  {TITLES[it][:80]}{mark}')
    print(f'  фактический следующий товар: {TITLES[S["target"][u]][:80]}')


# Где таргет попал во второй по массе кластер
second_hit = []
for u in np.flatnonzero(S['hit'])[:4000]:
    rank, mass = user_clusters(int(u))
    pos = int(np.flatnonzero(S['items'][u] == S['target'][u])[0])
    if rank[pos] == 1 and mass[1] >= M_MIN:
        second_hit.append(int(u))
    if len(second_hit) >= 3:
        break

masses2 = np.array([user_clusters(int(u))[1][1] for u in sub[:500]])
show_user(int(sub[int(np.argmin(masses2))]), 'одна доминирующая мода')
show_user(int(sub[int(np.argmax(masses2))]), 'две сопоставимые моды')
for u in second_hit[:2]:
    show_user(u, 'таргет во втором кластере — то, что теряет одна точка')

## 8. Выводы

*Заполнить после прогона.* Вывод строится вокруг пяти чисел:

1. **Доля качества, которую восстанавливает одна точка** (раздел 3): `k=1` против HR@10 самого
   SASRec в коллаборативном пространстве. Плюс форма лестницы: на каком k она выходит на плато.
2. **Абсолютный прирост от k = 1 к k = 3** и то, у каких пользователей он возникает (разбивка
   по exp(H); смотреть на абсолютную колонку, а не на отношение).
3. **Сколько прироста забирает MMR** — если большую часть, выигрыш объясняется избыточностью
   выдачи, а не модами.
4. **Разрыв между k = 1 и оракулом выбора моды** — потолок для генеративной модели, умеющей
   выбирать моду.
5. **Разделённость носителя против унимодального нуля** (cos между центроидами, отношение
   внутри/между): есть ли вообще моды, или носитель — одно облако.

Ориентиры этого датасета (тест, HR@10): SASRec 0.0445, текстовый kNN по последнему товару
0.0428, популярность 0.0101, случайный выбор 0.0005.